# Pretraining for ASR

In [1]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!{sys.executable} -m pip install datasets transformers tokenizers soundfile librosa jiwer evaluate

Looking in indexes: https://download.pytorch.org/whl/cu121


In [2]:
#installing libs
# !pip3 install torch torchvision torchaudio datasets transformers soundfile jiwer --index-url https://download.pytorch.org/whl/cu118
# !pip3 install librosa --index-url https://pypi.org/simple

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
import re
import torch
import torch.nn as nn
import numpy as np

from datasets import load_dataset, disable_caching
from transformers import Wav2Vec2ForPreTraining, Wav2Vec2FeatureExtractor, Wav2Vec2ForCTC, Wav2Vec2Processor
from transformers.models.wav2vec2.modeling_wav2vec2 import Wav2Vec2Encoder
import evaluate
metric = evaluate.load("accuracy")


True
1


## Finetuning Wav2Vec2 model on CTC loss (5 points)


In this task you have to create pipeline for finetuning pretrained multilingual Wav2Vec2 model on belarusian audio from [Fleurs](https://huggingface.co/datasets/google/fleurs) dataset.

#### Prepare data

In [4]:
from datasets import load_dataset, Audio

base_url = "https://huggingface.co/datasets/google/fleurs/resolve/refs%2Fconvert%2Fparquet/be_by"

data_files = {
    "train": f"{base_url}/train/0000.parquet",
    "validation": f"{base_url}/validation/0000.parquet",
    "test": f"{base_url}/test/0000.parquet"
}

fleurs = load_dataset("parquet", data_files=data_files)

# Must cast BEFORE any filter/map — prevents torchcodec from being called
fleurs = fleurs.cast_column("audio", Audio(decode=False))

print(fleurs)

DatasetDict({
    train: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 600
    })
    validation: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 408
    })
    test: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 600
    })
})


In this task, you should:

* filter all samples, where `transcription` includes digits. Hint: take care of specific belarussian symbols "і", "ў";
* remove punctuation from `transcription`.

In [5]:
def has_digits(text):
    return bool(re.search(r'\d', text))

def remove_punctuation(text):
    text = re.sub(r"[^\w\s'іўІЎ]", '', text, flags=re.UNICODE)
    text = re.sub(r'_', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def preprocess(sample):
    sample["transcription"] = remove_punctuation(sample["transcription"])
    return sample

preprocessed_train = fleurs["train"].filter(lambda x: not has_digits(x["transcription"])).map(preprocess)
preprocessed_val = fleurs["validation"].filter(lambda x: not has_digits(x["transcription"])).map(preprocess)

print("Sample transcription:", preprocessed_train[0]["transcription"])

Sample transcription: у той жа час паблізу ад верагодных маршрутаў уварвання базіравалася вельмі мала караблёў каралеўскага флоту таму што адміралы асцерагаліся іх патаплення нямецкімі паветранымі сіламі


#### Train tokenizer

There you should train your own BPE tokenizer based on texts from Fleurs dataset using [HuggingFace tokenizer](https://huggingface.co/docs/tokenizers/en/training_from_memory).

In [6]:
import os
os.environ["HF_DATASETS_OFFLINE"] = "0"
!pip install soundfile librosa

In [7]:
all_texts = (
    list(preprocessed_train["transcription"]) +
    list(preprocessed_val["transcription"])
)

all_chars = set()
for text in all_texts:
    all_chars.update(text)
all_chars = sorted(all_chars)

PAD_TOKEN = "[PAD]"
UNK_TOKEN = "[UNK]"
BOS_TOKEN = "[BOS]"
EOS_TOKEN = "[EOS]"
SPACE_TOKEN = "|"  

special_tokens = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]

vocab = {}
for tok in special_tokens:
    vocab[tok] = len(vocab)
vocab[SPACE_TOKEN] = len(vocab)
for ch in all_chars:
    if ch != " " and ch not in vocab:
        vocab[ch] = len(vocab)

id2token = {v: k for k, v in vocab.items()}

print(f"Char vocab size: {len(vocab)}")
print("Vocab:", list(vocab.keys()))

PAD_ID = vocab[PAD_TOKEN]

class CharTokenizer:
    """Simple character-level tokenizer using | as space token."""
    def __init__(self, vocab, id2token):
        self.vocab = vocab
        self.id2token = id2token

    def token_to_id(self, token):
        return self.vocab[token]

    def get_vocab_size(self):
        return len(self.vocab)

    def encode(self, text):
        ids = []
        for ch in text:
            if ch == " ":
                ids.append(self.vocab[SPACE_TOKEN])
            else:
                ids.append(self.vocab.get(ch, self.vocab[UNK_TOKEN]))
        return type("Enc", (), {"ids": ids})()

    def decode(self, ids):
        chars = []
        for i in ids:
            tok = self.id2token.get(i, "")
            if tok == SPACE_TOKEN:
                chars.append(" ")
            elif tok in (PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN):
                continue
            else:
                chars.append(tok)
        return "".join(chars)

tokenizer = CharTokenizer(vocab, id2token)

sample = all_texts[0]
enc = tokenizer.encode(sample)
dec = tokenizer.decode(enc.ids)
print("Original :", sample[:60])
print("Roundtrip:", dec[:60])
assert sample[:60] == dec[:60], "Roundtrip failed!"
print("Roundtrip OK")


Char vocab size: 61
Vocab: ['[PAD]', '[UNK]', '[BOS]', '[EOS]', '|', "'", 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'v', 'y', 'z', 'ú', 'а', 'б', 'в', 'г', 'д', 'е', 'ж', 'з', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 'ф', 'х', 'ц', 'ч', 'ш', 'ы', 'ь', 'э', 'ю', 'я', 'ё', 'і', 'ў']
Original : у той жа час паблізу ад верагодных маршрутаў уварвання базір
Roundtrip: у той жа час паблізу ад верагодных маршрутаў уварвання базір
Roundtrip OK


#### Loading model and preprocessor

In [8]:
from transformers import Wav2Vec2FeatureExtractor
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
   "facebook/wav2vec2-xls-r-300m"
)
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m", 
    ctc_loss_reduction="mean", 
    pad_token_id=tokenizer.token_to_id(PAD_TOKEN),
    vocab_size=tokenizer.get_vocab_size(),
    use_safetensors=True
)

model.freeze_feature_encoder()
print("Feature encoder frozen.")


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_q.weight             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Feature encoder frozen.


#### Data processor and data collator 

In [9]:
import io
import soundfile as sf
import librosa
import numpy as np

class CtcDataProcessor:
    def __init__(self, tokenizer, feature_extractor):
        self.tokenizer = tokenizer
        self.feature_extractor = feature_extractor

    def __call__(self, row):
        row["labels"] = self.tokenizer.encode(row["transcription"]).ids

        try:
            audio_data = row["audio"]["bytes"]
            
            with io.BytesIO(audio_data) as f:
                audio_array, sampling_rate = sf.read(f)
            
            if len(audio_array.shape) > 1:
                audio_array = np.mean(audio_array, axis=1)
                
            if sampling_rate != 16000:
                audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=16000)
            
            inputs = self.feature_extractor(
                audio_array, 
                sampling_rate=16000, 
                return_tensors="np"
            )
            row["input_values"] = inputs.input_values[0].tolist()
            
        except Exception as e:
            row["input_values"] = []
            
        return row

In [10]:
import io
import datasets
from datasets import Audio

preprocessed_train = preprocessed_train.cast_column("audio", Audio(decode=False))
preprocessed_val   = preprocessed_val.cast_column("audio",   Audio(decode=False))

sample = preprocessed_train[0]
print("audio keys:", list(sample["audio"].keys()))
print("bytes type:", type(sample["audio"]["bytes"]))


audio keys: ['bytes', 'path']
bytes type: <class 'bytes'>


In [11]:
from datasets import disable_caching
disable_caching()

data_processor = CtcDataProcessor(tokenizer, feature_extractor)

train_dataset = preprocessed_train.map(
    data_processor,
    remove_columns=preprocessed_train.column_names,
    keep_in_memory=False,
)

val_dataset = preprocessed_val.map(
    data_processor,
    remove_columns=preprocessed_val.column_names,
    keep_in_memory=False,
)

train_dataset = train_dataset.filter(lambda x: len(x["input_values"]) > 0)
val_dataset   = val_dataset.filter(lambda x: len(x["input_values"]) > 0)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")


Map:   0%|          | 0/481 [00:00<?, ? examples/s]

Map:   0%|          | 0/355 [00:00<?, ? examples/s]

Filter:   0%|          | 0/481 [00:00<?, ? examples/s]

Filter:   0%|          | 0/355 [00:00<?, ? examples/s]

Train samples: 481, Val samples: 355


In [12]:
class CTCDataCollator:
    LABELS_PAD_IDX = -100

    @staticmethod
    def collate_tokens(tokens_batch, type, pad_value=0.0):
        """
            Function collates list of tokens
        """
        max_len = max(len(t) for t in tokens_batch)
        if type == "input_values":
            padded = np.zeros((len(tokens_batch), max_len), dtype=np.float32)
            attention_mask = np.zeros((len(tokens_batch), max_len), dtype=np.int64)
            for i, t in enumerate(tokens_batch):
                padded[i, :len(t)] = t
                attention_mask[i, :len(t)] = 1
            return torch.tensor(padded), torch.tensor(attention_mask)
        else:  
            padded = np.full((len(tokens_batch), max_len), fill_value=pad_value, dtype=np.int64)
            for i, t in enumerate(tokens_batch):
                padded[i, :len(t)] = t
            return torch.tensor(padded)
        
    def __call__(self, batch):
        """
            Function collates `input_values` and `labels` into one tensor respectively
            Input: list with dicts, output of CTCDataProcessor
            Output row includes `labels` column with tokenized sequence, `input_values` column with computed spectrogram and 
            `attention_mask` (0 for not-attending position, 1 for attending)
        """
        input_values = [np.array(item["input_values"]) for item in batch]
        labels = [item["labels"] for item in batch]

        input_values_tensor, attention_mask = self.collate_tokens(input_values, type="input_values")
        labels_tensor = self.collate_tokens(labels, type="labels", pad_value=self.LABELS_PAD_IDX)

        return {
            "input_values": input_values_tensor,
            "attention_mask": attention_mask,
            "labels": labels_tensor,
        }

#### Inference and metrics computing

There you should use simple greedy straregy for CTC output decoding. 

Hint: Don't forget about padding value -100 in reference.

Hint: Don't forget about CTC output format.

In [13]:
!pip install jiwer

In [14]:
from itertools import groupby
import evaluate
wer_metric = evaluate.load("wer")

class MetricsComputer:
    def __call__(self, pred):
        """
            Input: object with fields `predictions` for CTC model output and `label_ids` for tokenized reference;
            Output: dict with key `wer` and computed wer
        """
        preds_logits = pred.predictions
        label_ids = pred.label_ids

        pred_ids = np.argmax(preds_logits, axis=-1)

        pad_token_id = tokenizer.token_to_id(PAD_TOKEN)

        def decode_ctc(token_ids):
            collapsed = [k for k, _ in groupby(token_ids)]
            filtered = [t for t in collapsed if t != pad_token_id]
            return tokenizer.decode(filtered)

        pred_str = [decode_ctc(ids) for ids in pred_ids]

        label_ids_clean = np.where(label_ids == -100, pad_token_id, label_ids)
        label_str = [
            tokenizer.decode([t for t in ids if t != pad_token_id])
            for ids in label_ids_clean
        ]

        print(f"Prediction: {pred_str[0]}")
        print(f"Reference:  {label_str[0]}")

        wer = wer_metric.compute(predictions=pred_str, references=label_str)
        return {"wer": wer}


#### Overfitting on train batch

In this task you should check pipeline correctness by overfitting on you need to finetune Wav2Vec2 model and achieve 50 WER or lower accuracy on val set.

In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="test",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    max_steps=3000,
    fp16=True,
    save_steps=500,
    eval_steps=100,
    logging_steps=50,
    # FIX: lower learning rate — 3e-4 was too aggressive, caused loss to plateau
    learning_rate=1e-4,
    weight_decay=0.005,
    # FIX: warmup_steps reduced from 500 to 100.
    # 500 warmup on 3000 steps = 17% warmup, which kept LR near zero too long
    # on a small dataset where the model starts diverging early.
    warmup_steps=100,
    gradient_checkpointing=True,
    # FIX: explicitly request GPU (cuda)
    dataloader_num_workers=0,
)

In [16]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
import os
print(os.environ.get("CUDA_VISIBLE_DEVICES"))


Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp311-cp311-linux_x86_64.whl (780.5 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp311-cp311-linux_x86_64.whl (7.3 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp311-cp311-linux_x86_64.whl (3.4 MB)
0


In [17]:
from transformers import Trainer
import torch

# FIX: assert GPU is available before training
assert torch.cuda.is_available(), "GPU not available! Training will be extremely slow on CPU."
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda")
model.to(device)

trainer = Trainer(
    model=model,
    data_collator=CTCDataCollator(),
    args=training_args,
    compute_metrics=MetricsComputer(),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)


CUDA available: True
GPU: NVIDIA H200


In [18]:
trainer.train()

Step,Training Loss,Validation Loss,Wer
100,20.946113,3.898579,1.000000
200,12.589423,3.227957,1.000000
300,12.609183,3.202785,1.000000
400,12.573679,3.192506,1.000000
500,12.354646,3.183890,1.000000
600,12.319459,3.149814,1.000000
700,9.294382,1.451984,0.999250
800,3.785011,0.656758,0.751912
900,2.555696,0.547802,0.637918
1000,1.955280,0.500787,0.588871


Prediction: 
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне
Prediction: 
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне
Prediction: 
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне
Prediction: 
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне
Prediction: 
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Prediction: 
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне
Prediction: чыліныадноітоічасупколтурычаставызначавцсвапрыналечна здаеепрасадмітны іцімвалічнустылівадзенніманірыпаводзціні чарооне
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне
Prediction: чылены адной і тойжа супкультуры часта вызначаюць сваю прыналежнаь здаее прас аадметныя сімвалічны стыль ўадзенні манерыпаводзен іхжаргоня
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне
Prediction: чылены адной і той жасупкультуры часта вызначаюць сваю прыналечнац здаее прасадметныі сімвалічных стылю адзенні манерыпаводзін ігхшарогоня
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе п

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Prediction: члены адной і той жа суб культуры часта вызначаюць сваю прыналежнась ьздаяе праз адметныя сімбалічны стылю аддзенні манерыпаводзін і жаргоння
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне
Prediction: члены адной і той жа суб культуры часта вызначаюць сваю прыналежнась здаяе праз адметны сімбалічны стылю аддзенні манерыпаводзін і жаргоння
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне
Prediction: члены адной і той жа суб культуры часта вызначаюць сваю прыналежнасць здаяе праз адметныя сімбалічны стылю аддзенні манерыпаводзін і жаргоння
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне
Prediction: члены адной і той жа суб культуры часта вызначаюць сваю прынале

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Prediction: члены адной і той жа суб культуры часта вызначаюць сваю прыналежнасць здаяе праз адметны сімбалічны стылю аддзенні манерыпаводзін і шжаргоння
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне
Prediction: члены адной і той жа суб культуры часта вызначаюць сваю прыналежнасць зда яе праз адметныя сімбалічны стылю аддзенні манерыпаводзін і шжаргоння
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне
Prediction: члены адной і той жа суб культуры часта вызначаюць сваю прыналежнасць здаяе праз адметныя сімбалічны стылю аддзенні манерыпаводзін і жаргоння
Reference:  члены адной і той жа субкультуры часта вызначаюць сваю прыналежнасць да яе праз адметны і сімвалічны стыль у адзенні манеры паводзін і жаргоне
Prediction: члены адной і той жа суб культуры часта вызначаюць сваю пры

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=4.594879024505615, metrics={'train_runtime': 6696.3415, 'train_samples_per_second': 7.168, 'train_steps_per_second': 0.448, 'total_flos': 2.608667030961412e+19, 'train_loss': 4.594879024505615, 'epoch': 96.79338842975207})